# Unsupervised Learning: Patient Phenotype Discovery

## Interactive Visualization of K-Means Clustering for Population Health

Welcome to this tutorial on **unsupervised learning** using patient phenotype discovery.
You'll watch clustering algorithms find natural patient cohorts with no labels at all -
exactly the problem a population health team faces when it is asked "who are our patients,
really?"

## What You'll Learn
- **Visual K-Means clustering**: watch centroids migrate to discover phenotypes
- **Pattern discovery**: the algorithm separates young and healthy, metabolic, frail elderly and complex chronic patients on its own
- **Mathematical intuition**: distance metrics and centroid calculations with clear explanations
- **Interactive exploration**: adjust k (number of phenotypes) and see immediate effects
- **Multiple algorithms**: K-Means, DBSCAN and hierarchical clustering comparisons
- **Practical application**: how discovered phenotypes shape a care pathway

> **About the data.** Every patient in this notebook is simulated from published clinical
> reference ranges. No real or identifiable patient data is used.

---


## 1. Introduction & Setup

**The Population Health Challenge:**

A care team planning services for a population needs to know which kinds of patient it is
actually serving, so it can target:
- **Intervention**: offer the right program to the right cohort
- **Capacity**: staff for the people who will actually turn up
- **Equity**: find the groups that current services are missing

**The Unsupervised Learning Solution:**

Rather than defining cohorts by hand from a few diagnosis codes, we'll use **clustering
algorithms** to discover natural groupings in routinely collected clinical measurements.
The algorithm will find structure we might miss.

**Key Difference from Supervised Learning:**
- ❌ **Supervised**: we provide labelled examples ("this patient is frail", "this one is metabolic")
- ✅ **Unsupervised**: the algorithm discovers the phenotypes on its own from unlabelled measurements


In [1]:
# Essential imports for patient phenotype clustering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, clear_output, HTML
import time

# Machine learning imports
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

# Interactive widgets
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

# Styling for clear clinical visualizations
plt.style.use('default')
sns.set_palette('Set2')
np.random.seed(42)

print("\U0001fa7a Patient Phenotype Discovery: Unsupervised Learning Ready")
print("   Demonstrating K-Means, DBSCAN, and Hierarchical Clustering")
print("   Application: population health cohort discovery and care pathway design")


🩺 Patient Phenotype Discovery: Unsupervised Learning Ready
   Demonstrating K-Means, DBSCAN, and Hierarchical Clustering
   Application: population health cohort discovery and care pathway design


## 2. Synthetic Patient Dataset Generation

Let's create a realistic dataset representing four clinical phenotypes. All values are
simulated from published reference ranges; no real or identifiable patient data is used.

**Phenotype 1 - Young and Healthy** (low age, almost no comorbidity):
- Age 20-38 years
- Comorbidity burden 0-1 conditions
- HbA1c 4.8-5.6 %
- eGFR 95-120 mL/min/1.73m²
- BMI 20-27, 0-3 visits a year

**Phenotype 2 - Metabolic** (raised BMI and HbA1c, middle-aged):
- Age 44-60 years
- Comorbidity burden 1-3 conditions
- HbA1c 7.0-8.8 % (diabetes range)
- eGFR 70-95 mL/min/1.73m²
- BMI 29-40, 3-7 visits a year

**Phenotype 3 - Frail Elderly** (high age, high comorbidity, low functional status):
- Age 75-92 years
- Comorbidity burden 3-6 conditions
- HbA1c 5.7-7.1 %
- eGFR 35-60 mL/min/1.73m² (stage 3 kidney disease)
- BMI 18-25, 5-10 visits a year

**Phenotype 4 - Complex Chronic** (high utilization, several organ systems):
- Age 55-78 years
- Comorbidity burden 5-9 conditions
- HbA1c 6.0-8.4 %
- eGFR 25-52 mL/min/1.73m² (stage 3b-4 kidney disease)
- BMI 24-35, 10-20 visits a year


In [2]:
def generate_patient_phenotype_data(n_samples=600, add_noise=True):
    """
    Generate realistic synthetic patient characteristic data.

    Features:
    - age: Age in years
    - comorbidity_burden: Number of active long-term conditions
    - hba1c: Glycated hemoglobin (%)
    - egfr: Estimated glomerular filtration rate (mL/min/1.73m^2)
    - bmi: Body mass index (kg/m^2)
    - annual_visits: Healthcare contacts in the last 12 months

    All values are simulated from published reference ranges. No real or
    identifiable patient data is used.
    """
    np.random.seed(42)

    samples_per_phenotype = n_samples // 4

    # Phenotype 1: Young and healthy (low age, minimal comorbidity)
    young_healthy = pd.DataFrame({
        'age': np.random.normal(28, 6, samples_per_phenotype),
        'comorbidity_burden': np.random.normal(0.4, 0.4, samples_per_phenotype),
        'hba1c': np.random.normal(5.2, 0.3, samples_per_phenotype),
        'egfr': np.random.normal(106, 10, samples_per_phenotype),
        'bmi': np.random.normal(23.5, 2.5, samples_per_phenotype),
        'annual_visits': np.random.normal(1.2, 0.9, samples_per_phenotype)
    })

    # Phenotype 2: Metabolic (raised BMI and HbA1c, middle-aged)
    metabolic = pd.DataFrame({
        'age': np.random.normal(52, 8, samples_per_phenotype),
        'comorbidity_burden': np.random.normal(2.2, 0.8, samples_per_phenotype),
        'hba1c': np.random.normal(7.8, 0.9, samples_per_phenotype),
        'egfr': np.random.normal(82, 12, samples_per_phenotype),
        'bmi': np.random.normal(34, 4, samples_per_phenotype),
        'annual_visits': np.random.normal(4.6, 1.6, samples_per_phenotype)
    })

    # Phenotype 3: Frail elderly (high age, high comorbidity, low functional status)
    frail_elderly = pd.DataFrame({
        'age': np.random.normal(83, 6, samples_per_phenotype),
        'comorbidity_burden': np.random.normal(4.4, 1.0, samples_per_phenotype),
        'hba1c': np.random.normal(6.4, 0.7, samples_per_phenotype),
        'egfr': np.random.normal(48, 12, samples_per_phenotype),
        'bmi': np.random.normal(21.5, 3.0, samples_per_phenotype),
        'annual_visits': np.random.normal(7.2, 2.2, samples_per_phenotype)
    })

    # Phenotype 4: Complex chronic (high utilization, multiple organ systems)
    complex_chronic = pd.DataFrame({
        'age': np.random.normal(66, 9, samples_per_phenotype),
        'comorbidity_burden': np.random.normal(6.8, 1.2, samples_per_phenotype),
        'hba1c': np.random.normal(7.2, 1.1, samples_per_phenotype),
        'egfr': np.random.normal(38, 13, samples_per_phenotype),
        'bmi': np.random.normal(29, 5, samples_per_phenotype),
        'annual_visits': np.random.normal(14.5, 4.0, samples_per_phenotype)
    })

    # Combine all phenotypes
    patient_data = pd.concat([young_healthy, metabolic, frail_elderly, complex_chronic],
                             ignore_index=True)

    # Add small amount of realistic noise if requested
    if add_noise:
        noise_scale = 0.05  # 5% noise
        for col in ['age', 'comorbidity_burden', 'hba1c', 'egfr', 'bmi', 'annual_visits']:
            noise = np.random.normal(0, patient_data[col].std() * noise_scale, len(patient_data))
            patient_data[col] += noise

    # Keep every value inside a clinically possible range
    patient_data['age'] = patient_data['age'].clip(18, 100)
    patient_data['comorbidity_burden'] = patient_data['comorbidity_burden'].clip(0, 12)
    patient_data['hba1c'] = patient_data['hba1c'].clip(4.0, 15.0)
    patient_data['egfr'] = patient_data['egfr'].clip(5, 130)
    patient_data['bmi'] = patient_data['bmi'].clip(14, 60)
    patient_data['annual_visits'] = patient_data['annual_visits'].clip(0, 40)

    # Add ground truth labels (only for evaluation - NOT used in clustering!)
    patient_data['true_phenotype'] = (['Young and Healthy'] * samples_per_phenotype +
                                      ['Metabolic'] * samples_per_phenotype +
                                      ['Frail Elderly'] * samples_per_phenotype +
                                      ['Complex Chronic'] * samples_per_phenotype)

    return patient_data

# Generate patient dataset
patient_data = generate_patient_phenotype_data(n_samples=600)

print("\U0001fa7a Patient Cohort Generated (simulated, no real patient data)")
print(f"   Total patients: {len(patient_data)}")
print(f"   Features: {patient_data.columns.tolist()[:-1]}")
print(f"   (True phenotypes included only for validation - NOT used in clustering)")

# Display basic statistics
print("\n\U0001f4ca Dataset Summary:")
display(patient_data.describe().round(2))

print("\n\U0001f9ec Cohort Coverage:")
print(f"   Age range: {patient_data['age'].min():.0f} to {patient_data['age'].max():.0f} years")
print(f"   Comorbidity burden: {patient_data['comorbidity_burden'].min():.1f} to "
      f"{patient_data['comorbidity_burden'].max():.1f} conditions")


🩺 Patient Cohort Generated (simulated, no real patient data)
   Total patients: 600
   Features: ['age', 'comorbidity_burden', 'hba1c', 'egfr', 'bmi', 'annual_visits']
   (True phenotypes included only for validation - NOT used in clustering)

📊 Dataset Summary:


,age,comorbidity_burden,hba1c,egfr,bmi,annual_visits
count,600.00,600.00,600.00,600.00,600.00,600.00
mean,57.46,3.45,6.63,68.56,27.17,6.97
std,21.50,2.54,1.24,29.72,6.23,5.52
min,18.00,0.00,4.05,5.00,15.30,0.00
25%,38.39,1.03,5.49,43.56,22.36,2.67
50%,59.41,3.16,6.59,65.10,25.77,5.59
75%,77.16,5.42,7.60,96.61,31.77,9.58
max,97.99,11.49,10.11,130.00,46.82,24.73



🧬 Cohort Coverage:
   Age range: 18 to 98 years
   Comorbidity burden: 0.0 to 11.5 conditions


In [3]:
# Visualize the patient dataset
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Phenotype Plane (Comorbidity vs Age)',
        'HbA1c vs BMI',
        'Kidney Function vs Age',
        'eGFR Distribution',
        'Annual Visit Bands',
        'Feature Correlations'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
        [{'type': 'box'}, {'type': 'bar'}, {'type': 'heatmap'}]
    ]
)

# Color by true phenotype (just for initial visualization)
phenotype_colors = {'Young and Healthy': '#3498db', 'Metabolic': '#e67e22',
                    'Frail Elderly': '#9b59b6', 'Complex Chronic': '#e74c3c'}

# Plot 1: The phenotype plane - comorbidity burden against age
for phenotype, color in phenotype_colors.items():
    subset = patient_data[patient_data['true_phenotype'] == phenotype]
    fig.add_trace(
        go.Scatter(
            x=subset['comorbidity_burden'],
            y=subset['age'],
            mode='markers',
            name=phenotype,
            marker=dict(size=5, color=color, opacity=0.6),
            showlegend=True
        ), row=1, col=1
    )

# Plot 2: HbA1c vs BMI
for phenotype, color in phenotype_colors.items():
    subset = patient_data[patient_data['true_phenotype'] == phenotype]
    fig.add_trace(
        go.Scatter(
            x=subset['bmi'],
            y=subset['hba1c'],
            mode='markers',
            name=phenotype,
            marker=dict(size=5, color=color, opacity=0.6),
            showlegend=False
        ), row=1, col=2
    )

# Plot 3: Kidney function against age
for phenotype, color in phenotype_colors.items():
    subset = patient_data[patient_data['true_phenotype'] == phenotype]
    fig.add_trace(
        go.Scatter(
            x=subset['age'],
            y=subset['egfr'],
            mode='markers',
            name=phenotype,
            marker=dict(size=5, color=color, opacity=0.6),
            showlegend=False
        ), row=1, col=3
    )

# Plot 4: eGFR distribution by phenotype
for phenotype, color in phenotype_colors.items():
    subset = patient_data[patient_data['true_phenotype'] == phenotype]
    fig.add_trace(
        go.Box(
            y=subset['egfr'],
            name=phenotype,
            marker_color=color,
            showlegend=False
        ), row=2, col=1
    )

# Plot 5: Annual visit bands
visit_bins = [0, 2, 4, 6, 10, 15, 100]
visit_labels = ['0-1', '2-3', '4-5', '6-9', '10-14', '15+']
visit_hist, _ = np.histogram(patient_data['annual_visits'], bins=visit_bins)

fig.add_trace(
    go.Bar(
        x=visit_labels,
        y=visit_hist,
        marker_color='#3498db',
        showlegend=False
    ), row=2, col=2
)

# Plot 6: Feature correlation heatmap
feature_cols = ['age', 'comorbidity_burden', 'hba1c', 'egfr', 'bmi', 'annual_visits']
correlation_matrix = patient_data[feature_cols].corr()

fig.add_trace(
    go.Heatmap(
        z=correlation_matrix.values,
        x=feature_cols,
        y=feature_cols,
        colorscale='RdBu',
        zmid=0,
        text=correlation_matrix.values.round(2),
        texttemplate='%{text}',
        showscale=False
    ), row=2, col=3
)

# Update layout
fig.update_layout(
    title='Patient Cohort Overview (Colors show true phenotypes - for reference only)',
    width=1400,
    height=800,
    showlegend=True
)

# Update axes
fig.update_xaxes(title_text="Comorbidity burden (conditions)", row=1, col=1)
fig.update_yaxes(title_text="Age (years)", row=1, col=1)
fig.update_xaxes(title_text="BMI (kg/m²)", row=1, col=2)
fig.update_yaxes(title_text="HbA1c (%)", row=1, col=2)
fig.update_xaxes(title_text="Age (years)", row=1, col=3)
fig.update_yaxes(title_text="eGFR (mL/min/1.73m²)", row=1, col=3)
fig.update_yaxes(title_text="eGFR (mL/min/1.73m²)", row=2, col=1)
fig.update_xaxes(title_text="Visits per year", row=2, col=2)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.show()

print("\n\U0001f50d Key Observations:")
print("   • The cohort shows natural clustering (but we'll pretend we don't know the phenotypes!)")
print("   • Age and comorbidity burden separate the groups most cleanly")
print("   • Kidney function falls steeply with age and comorbidity")
print("   • Perfect scenario for unsupervised learning to discover patterns")



🔍 Key Observations:
   • The cohort shows natural clustering (but we'll pretend we don't know the phenotypes!)
   • Age and comorbidity burden separate the groups most cleanly
   • Kidney function falls steeply with age and comorbidity
   • Perfect scenario for unsupervised learning to discover patterns


## 3. K-Means Clustering Fundamentals

### How K-Means Works

K-Means is an **iterative algorithm** that discovers k clusters by:

1. **Initialize**: Randomly place k centroids (cluster centers)
2. **Assign**: Assign each data point to the nearest centroid
3. **Update**: Move each centroid to the mean of its assigned points
4. **Repeat**: Steps 2-3 until centroids stop moving (convergence)

### Mathematical Foundation

**Distance Metric** (Euclidean distance):
```
d(p, c) = √[(x₁-cx₁)² + (x₂-cx₂)² + ... + (xₙ-cxₙ)²]
```

**Objective**: Minimize within-cluster sum of squares (inertia):
```
Inertia = Σ Σ ||x - μₖ||²
         k x∈Cₖ
```

Where:
- k = cluster index
- Cₖ = set of points in cluster k
- μₖ = centroid of cluster k
- x = data point

### Why K-Means for Patient Phenotypes?

✅ **Efficient**: fast even across a whole registered population
✅ **Interpretable**: a centroid is a description of the "typical" patient in that cohort
✅ **Scalable**: works with many clinical features at once
✅ **Practical**: a newly registered patient is assigned to a cohort in one step


In [4]:
# Demonstrate K-Means concept with simple 2D example
def visualize_kmeans_concept():
    """
    Simple visualization of K-Means concept on the phenotype plane
    (comorbidity burden against age).
    """
    # Use just 2 features for easy visualization
    X_simple = patient_data[['comorbidity_burden', 'age']].values

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_simple)

    # Fit K-Means with k=4
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)

    # Transform centroids back to original scale for interpretation
    centroids_scaled = kmeans.cluster_centers_
    centroids_original = scaler.inverse_transform(centroids_scaled)

    # Create visualization
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('K-Means Discovered Cohorts', 'Cluster Centroids (Typical Patient)')
    )

    # Plot 1: Data points colored by discovered cluster
    colors = px.colors.qualitative.Set2[:4]

    for cluster_id in range(4):
        mask = clusters == cluster_id
        fig.add_trace(
            go.Scatter(
                x=patient_data.loc[mask, 'comorbidity_burden'],
                y=patient_data.loc[mask, 'age'],
                mode='markers',
                name=f'Phenotype {cluster_id+1}',
                marker=dict(size=6, color=colors[cluster_id], opacity=0.6)
            ), row=1, col=1
        )

    # Add centroids
    fig.add_trace(
        go.Scatter(
            x=centroids_original[:, 0],
            y=centroids_original[:, 1],
            mode='markers',
            name='Centroids',
            marker=dict(size=20, color='red', symbol='star',
                       line=dict(color='black', width=2))
        ), row=1, col=1
    )

    # Plot 2: Centroid characteristics
    cluster_names = [f'Phenotype {i+1}' for i in range(4)]

    fig.add_trace(
        go.Scatter(
            x=centroids_original[:, 0],
            y=centroids_original[:, 1],
            mode='markers+text',
            name='Centroids',
            marker=dict(size=25, color=colors,
                       line=dict(color='black', width=2)),
            text=cluster_names,
            textposition='top center',
            textfont=dict(size=12, color='black'),
            showlegend=False
        ), row=1, col=2
    )

    # Update layout
    fig.update_layout(
        title='K-Means Clustering: Discovering Phenotypes from Comorbidity Burden & Age',
        width=1400,
        height=600,
        showlegend=True
    )

    fig.update_xaxes(title_text="Comorbidity burden (conditions)", row=1, col=1)
    fig.update_yaxes(title_text="Age (years)", row=1, col=1)
    fig.update_xaxes(title_text="Comorbidity burden (conditions)", row=1, col=2)
    fig.update_yaxes(title_text="Age (years)", row=1, col=2)

    fig.show()

    # Print centroid interpretation
    print("\U0001f3af Discovered Phenotype Characteristics:")
    print("=" * 60)
    for i, (burden, age) in enumerate(centroids_original):
        print(f"\nPhenotype {i+1}:")
        print(f"  Average comorbidity burden: {burden:.2f} conditions")
        print(f"  Average age: {age:.1f} years")

        # Interpret the cohort based on its centroid
        if burden > 5.5:
            cohort_type = "Complex chronic"
        elif age > 75:
            cohort_type = "Frail elderly"
        elif burden > 1.4:
            cohort_type = "Metabolic / early multimorbidity"
        else:
            cohort_type = "Young and healthy"

        print(f"  Interpretation: {cohort_type}")

    print("\n\U0001f4a1 Key Insight: the algorithm found these cohorts WITHOUT being told what to look for!")

visualize_kmeans_concept()


🎯 Discovered Phenotype Characteristics:

Phenotype 1:
  Average comorbidity burden: 4.39 conditions
  Average age: 82.3 years
  Interpretation: Frail elderly

Phenotype 2:
  Average comorbidity burden: 0.46 conditions
  Average age: 27.8 years
  Interpretation: Young and healthy

Phenotype 3:
  Average comorbidity burden: 7.01 conditions
  Average age: 65.9 years
  Interpretation: Complex chronic

Phenotype 4:
  Average comorbidity burden: 2.29 conditions
  Average age: 53.1 years
  Interpretation: Metabolic / early multimorbidity

💡 Key Insight: the algorithm found these cohorts WITHOUT being told what to look for!


## 4. Animated K-Means Training Visualization

Now for the interesting part. Watch the K-Means algorithm iteratively discover patient
phenotypes by moving centroids to optimize cluster assignments.

This animation shows:
- **Initial random centroids** (iteration 0)
- **Centroid migration paths** across iterations
- **Cluster assignment changes** as centroids move
- **Convergence** when centroids stabilize


In [5]:
class AnimatedKMeans:
    """
    Custom K-Means implementation with iteration tracking for animation.
    """
    
    def __init__(self, n_clusters=4, max_iterations=50, random_state=42):
        self.n_clusters = n_clusters
        self.max_iterations = max_iterations
        self.random_state = random_state
        self.history = []
    
    def fit(self, X):
        """Fit K-Means and record history for animation."""
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        
        # Initialize centroids randomly from data points
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        centroids = X[random_indices].copy()
        
        # Track initial state
        labels = self._assign_clusters(X, centroids)
        inertia = self._compute_inertia(X, centroids, labels)
        self.history.append({
            'iteration': 0,
            'centroids': centroids.copy(),
            'labels': labels.copy(),
            'inertia': inertia
        })
        
        # Iterative optimization
        for iteration in range(1, self.max_iterations + 1):
            # Assign points to nearest centroid
            labels = self._assign_clusters(X, centroids)
            
            # Update centroids
            new_centroids = np.array([
                X[labels == k].mean(axis=0) if np.sum(labels == k) > 0 else centroids[k]
                for k in range(self.n_clusters)
            ])
            
            # Compute inertia
            inertia = self._compute_inertia(X, new_centroids, labels)
            
            # Store history
            self.history.append({
                'iteration': iteration,
                'centroids': new_centroids.copy(),
                'labels': labels.copy(),
                'inertia': inertia
            })
            
            # Check convergence
            if np.allclose(centroids, new_centroids, rtol=1e-4):
                print(f"   Converged at iteration {iteration}")
                break
            
            centroids = new_centroids
        
        self.centroids = centroids
        self.labels = labels
        self.inertia = inertia
        
        return self
    
    def _assign_clusters(self, X, centroids):
        """Assign each point to nearest centroid."""
        distances = cdist(X, centroids, metric='euclidean')
        return np.argmin(distances, axis=1)
    
    def _compute_inertia(self, X, centroids, labels):
        """Compute within-cluster sum of squares."""
        return np.sum([
            np.sum((X[labels == k] - centroids[k]) ** 2)
            for k in range(self.n_clusters)
        ])

def animate_kmeans_training(n_clusters=4, update_frequency=3):
    """
    Animate K-Means training process showing centroid movement.
    """
    # Use the phenotype plane (comorbidity burden, age) for 2D visualization
    X_simple = patient_data[['comorbidity_burden', 'age']].values
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_simple)
    
    # Train with history tracking
    print("🚀 Starting Animated K-Means Training...")
    print(f"   Number of clusters: {n_clusters}")
    print(f"   Watch centroids migrate to discover patient phenotypes!\n")
    
    kmeans = AnimatedKMeans(n_clusters=n_clusters, max_iterations=30, random_state=42)
    kmeans.fit(X_scaled)
    
    # Create animation frames
    colors = px.colors.qualitative.Set2[:n_clusters]
    
    for idx, state in enumerate(kmeans.history):
        if idx % update_frequency != 0 and idx != len(kmeans.history) - 1:
            continue
        
        clear_output(wait=True)
        
        # Transform centroids back to original scale
        centroids_original = scaler.inverse_transform(state['centroids'])
        labels = state['labels']
        iteration = state['iteration']
        inertia = state['inertia']
        
        # Create visualization
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.65, 0.35],
            subplot_titles=(
                f'Iteration {iteration}: Cluster Discovery',
                'Inertia Reduction (Optimization Progress)'
            )
        )
        
        # Plot 1: Data points with cluster assignments
        for cluster_id in range(n_clusters):
            mask = labels == cluster_id
            fig.add_trace(
                go.Scatter(
                    x=patient_data.loc[mask, 'comorbidity_burden'],
                    y=patient_data.loc[mask, 'age'],
                    mode='markers',
                    name=f'Phenotype {cluster_id+1}',
                    marker=dict(size=6, color=colors[cluster_id], opacity=0.5),
                    showlegend=True
                ), row=1, col=1
            )
        
        # Add current centroids
        fig.add_trace(
            go.Scatter(
                x=centroids_original[:, 0],
                y=centroids_original[:, 1],
                mode='markers',
                name='Centroids',
                marker=dict(size=20, color='red', symbol='star',
                           line=dict(color='black', width=2)),
                showlegend=True
            ), row=1, col=1
        )
        
        # Add centroid trajectories (show last few iterations)
        if idx > 0:
            for cluster_id in range(n_clusters):
                trajectory_x = []
                trajectory_y = []
                
                # Get trajectory for this cluster
                start_idx = max(0, idx - 5)
                for hist_idx in range(start_idx, idx + 1):
                    hist_centroids = scaler.inverse_transform(kmeans.history[hist_idx]['centroids'])
                    trajectory_x.append(hist_centroids[cluster_id, 0])
                    trajectory_y.append(hist_centroids[cluster_id, 1])
                
                # Plot trajectory
                fig.add_trace(
                    go.Scatter(
                        x=trajectory_x,
                        y=trajectory_y,
                        mode='lines',
                        name=f'Path {cluster_id+1}',
                        line=dict(color=colors[cluster_id], width=2, dash='dot'),
                        showlegend=False,
                        opacity=0.7
                    ), row=1, col=1
                )
        
        # Plot 2: Inertia over iterations
        inertias = [h['inertia'] for h in kmeans.history[:idx+1]]
        iterations_so_far = [h['iteration'] for h in kmeans.history[:idx+1]]
        
        fig.add_trace(
            go.Scatter(
                x=iterations_so_far,
                y=inertias,
                mode='lines+markers',
                name='Inertia',
                line=dict(color='blue', width=3),
                marker=dict(size=8),
                showlegend=False
            ), row=1, col=2
        )
        
        # Highlight current point
        fig.add_trace(
            go.Scatter(
                x=[iteration],
                y=[inertia],
                mode='markers',
                name='Current',
                marker=dict(size=15, color='red', symbol='circle'),
                showlegend=False
            ), row=1, col=2
        )
        
        # Update layout
        fig.update_layout(
            title=f'K-Means Training Animation | Iteration {iteration}/{kmeans.history[-1]["iteration"]} | Inertia: {inertia:.2f}',
            width=1400,
            height=600,
            showlegend=True
        )
        
        fig.update_xaxes(title_text="Comorbidity burden (conditions)", row=1, col=1)
        fig.update_yaxes(title_text="Age (years)", row=1, col=1)
        fig.update_xaxes(title_text="Iteration", row=1, col=2)
        fig.update_yaxes(title_text="Inertia (Within-Cluster SS)", row=1, col=2)
        
        fig.show()
        
        print(f"Iteration {iteration:2d}: Inertia = {inertia:.4f}")
        
        # Small delay for animation effect
        if idx < len(kmeans.history) - 1:
            time.sleep(0.3)
    
    print("\n🎯 K-Means Converged!")
    print(f"   Final Inertia: {kmeans.history[-1]['inertia']:.4f}")
    print(f"   Patient phenotypes successfully discovered through unsupervised learning!")
    
    return kmeans, scaler

# Run the animation
kmeans_animated, scaler = animate_kmeans_training(n_clusters=4, update_frequency=2)

Iteration  7: Inertia = 130.5890

🎯 K-Means Converged!
   Final Inertia: 130.5890
   Patient phenotypes successfully discovered through unsupervised learning!


## 5. Interactive Parameter Exploration

Let's explore how the number of clusters (k) affects what we discover. Try different
values and see how the population is divided into phenotypes.


In [6]:
def interactive_kmeans_exploration(n_clusters=4):
    """
    Interactive K-Means with adjustable number of phenotypes.
    """
    # Prepare full feature set (not just comorbidity burden & age)
    feature_cols = ['age', 'comorbidity_burden', 'hba1c', 'egfr']
    X = patient_data[feature_cols].values

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fit K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)

    # Get centroids in original scale
    centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)

    # Create comprehensive visualization
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f'Phenotype Plane ({n_clusters} Cohorts)',
            f'HbA1c vs Kidney Function ({n_clusters} Cohorts)',
            f'Cohort Sizes',
            f'Centroid Characteristics'
        ),
        specs=[
            [{'type': 'scatter'}, {'type': 'scatter'}],
            [{'type': 'bar'}, {'type': 'bar'}]
        ]
    )

    colors = px.colors.qualitative.Set2[:n_clusters]

    # Plot 1: The phenotype plane - comorbidity burden against age
    for cluster_id in range(n_clusters):
        mask = clusters == cluster_id
        fig.add_trace(
            go.Scatter(
                x=patient_data.loc[mask, 'comorbidity_burden'],
                y=patient_data.loc[mask, 'age'],
                mode='markers',
                name=f'Phenotype {cluster_id+1}',
                marker=dict(size=5, color=colors[cluster_id], opacity=0.6)
            ), row=1, col=1
        )

    # Add centroids to plot 1
    fig.add_trace(
        go.Scatter(
            x=centroids_original[:, 1],  # comorbidity burden
            y=centroids_original[:, 0],  # age
            mode='markers+text',
            name='Centroids',
            marker=dict(size=20, color='red', symbol='star',
                       line=dict(color='black', width=2)),
            text=[f'{i+1}' for i in range(n_clusters)],
            textfont=dict(color='white', size=12),
            showlegend=False
        ), row=1, col=1
    )

    # Plot 2: HbA1c against kidney function
    for cluster_id in range(n_clusters):
        mask = clusters == cluster_id
        fig.add_trace(
            go.Scatter(
                x=patient_data.loc[mask, 'egfr'],
                y=patient_data.loc[mask, 'hba1c'],
                mode='markers',
                name=f'Phenotype {cluster_id+1}',
                marker=dict(size=5, color=colors[cluster_id], opacity=0.6),
                showlegend=False
            ), row=1, col=2
        )

    # Add centroids to plot 2
    fig.add_trace(
        go.Scatter(
            x=centroids_original[:, 3],  # eGFR
            y=centroids_original[:, 2],  # HbA1c
            mode='markers+text',
            name='Centroids',
            marker=dict(size=20, color='red', symbol='star',
                       line=dict(color='black', width=2)),
            text=[f'{i+1}' for i in range(n_clusters)],
            textfont=dict(color='white', size=12),
            showlegend=False
        ), row=1, col=2
    )

    # Plot 3: Cohort sizes
    cluster_sizes = [np.sum(clusters == i) for i in range(n_clusters)]
    fig.add_trace(
        go.Bar(
            x=[f'Phenotype {i+1}' for i in range(n_clusters)],
            y=cluster_sizes,
            marker_color=colors,
            text=cluster_sizes,
            textposition='auto',
            showlegend=False
        ), row=2, col=1
    )

    # Plot 4: Centroid characteristics (grouped bars for multiple features)
    feature_names = ['Age (yr)', 'Comorbidity (n)', 'HbA1c (%)', 'eGFR (mL/min)']

    # Normalize centroids for comparison (0-1 scale for each feature)
    centroids_normalized = (centroids_original - centroids_original.min(axis=0)) / \
                          (centroids_original.max(axis=0) - centroids_original.min(axis=0) + 1e-10)

    for feature_idx, feature_name in enumerate(feature_names):
        fig.add_trace(
            go.Bar(
                x=[f'Phenotype {i+1}' for i in range(n_clusters)],
                y=centroids_normalized[:, feature_idx],
                name=feature_name,
                showlegend=True
            ), row=2, col=2
        )

    # Update layout
    fig.update_layout(
        title=f'K-Means Clustering with k={n_clusters} | Inertia: {kmeans.inertia_:.2f}',
        width=1400,
        height=800,
        barmode='group'
    )

    fig.update_xaxes(title_text="Comorbidity burden (conditions)", row=1, col=1)
    fig.update_yaxes(title_text="Age (years)", row=1, col=1)
    fig.update_xaxes(title_text="eGFR (mL/min/1.73m²)", row=1, col=2)
    fig.update_yaxes(title_text="HbA1c (%)", row=1, col=2)
    fig.update_xaxes(title_text="Cluster", row=2, col=1)
    fig.update_yaxes(title_text="Number of Patients", row=2, col=1)
    fig.update_xaxes(title_text="Cluster", row=2, col=2)
    fig.update_yaxes(title_text="Normalized Value", row=2, col=2)

    fig.show()

    # Print cluster characteristics
    print(f"\n\U0001fa7a Discovered Phenotype Characteristics (k={n_clusters}):")
    print("=" * 80)

    for cluster_id in range(n_clusters):
        print(f"\nPhenotype {cluster_id+1}:")
        print(f"  Size: {cluster_sizes[cluster_id]} patients")
        print(f"  Avg age: {centroids_original[cluster_id, 0]:.1f} years")
        print(f"  Avg comorbidity burden: {centroids_original[cluster_id, 1]:.2f} conditions")
        print(f"  Avg HbA1c: {centroids_original[cluster_id, 2]:.2f} %")
        print(f"  Avg eGFR: {centroids_original[cluster_id, 3]:.0f} mL/min/1.73m2")

    # Silhouette score for cluster quality
    silhouette_avg = silhouette_score(X_scaled, clusters)
    print(f"\n\U0001f4ca Cluster Quality Metrics:")
    print(f"   Silhouette Score: {silhouette_avg:.3f} (closer to 1 is better)")
    print(f"   Inertia: {kmeans.inertia_:.2f} (lower is better)")

# Create interactive widget
interact(interactive_kmeans_exploration,
         n_clusters=IntSlider(value=4, min=2, max=8, step=1,
                             description='Number of Phenotypes (k)'));


interactive(children=(IntSlider(value=4, description='Number of Phenotypes (k)', max=8, min=2), Output()), _do…

## 6. Finding Optimal k: Elbow Method & Silhouette Analysis

One of the biggest challenges in unsupervised learning: **How many phenotypes should we use?**

We'll use two methods:
1. **Elbow Method**: Plot inertia vs k, look for the "elbow" where adding clusters gives diminishing returns
2. **Silhouette Analysis**: Measures how well patients fit in their assigned cohort (higher is better)


In [7]:
# Prepare data for optimal k analysis
feature_cols = ['age', 'comorbidity_burden', 'hba1c', 'egfr']
X = patient_data[feature_cols].values
scaler_full = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X)

# Test different k values
k_range = range(2, 11)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []

print("🔍 Evaluating different numbers of clusters...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled_full)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled_full, clusters))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled_full, clusters))
    
    print(f"   k={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.3f}")

# Create comprehensive visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Elbow Method: Inertia vs Number of Clusters',
        'Silhouette Score vs Number of Clusters',
        'Davies-Bouldin Index (Lower is Better)',
        'Silhouette Analysis for k=4'
    )
)

# Plot 1: Elbow curve
fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=inertias,
        mode='lines+markers',
        name='Inertia',
        line=dict(color='blue', width=3),
        marker=dict(size=10),
        showlegend=False
    ), row=1, col=1
)

# Highlight potential elbow (k=4 in this case)
fig.add_trace(
    go.Scatter(
        x=[4],
        y=[inertias[2]],  # index 2 = k=4
        mode='markers',
        name='Suggested k=4',
        marker=dict(size=20, color='red', symbol='star'),
        showlegend=False
    ), row=1, col=1
)

# Plot 2: Silhouette scores
fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=silhouette_scores,
        mode='lines+markers',
        name='Silhouette Score',
        line=dict(color='green', width=3),
        marker=dict(size=10),
        showlegend=False
    ), row=1, col=2
)

# Highlight best silhouette score
best_k_silhouette = k_range[np.argmax(silhouette_scores)]
fig.add_trace(
    go.Scatter(
        x=[best_k_silhouette],
        y=[max(silhouette_scores)],
        mode='markers',
        name=f'Best k={best_k_silhouette}',
        marker=dict(size=20, color='red', symbol='star'),
        showlegend=False
    ), row=1, col=2
)

# Plot 3: Davies-Bouldin Index
fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=davies_bouldin_scores,
        mode='lines+markers',
        name='Davies-Bouldin',
        line=dict(color='orange', width=3),
        marker=dict(size=10),
        showlegend=False
    ), row=2, col=1
)

# Plot 4: Detailed silhouette analysis for k=4
kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_4 = kmeans_4.fit_predict(X_scaled_full)
silhouette_vals = silhouette_samples(X_scaled_full, clusters_4)

y_lower = 10
colors_sil = px.colors.qualitative.Set2[:4]

for i in range(4):
    # Get silhouette values for cluster i
    cluster_silhouette_vals = silhouette_vals[clusters_4 == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    fig.add_trace(
        go.Bar(
            x=cluster_silhouette_vals,
            y=list(range(y_lower, y_upper)),
            orientation='h',
            name=f'Phenotype {i+1}',
            marker=dict(color=colors_sil[i]),
            showlegend=True
        ), row=2, col=2
    )
    
    y_lower = y_upper + 10

# Add average silhouette score line
avg_silhouette = silhouette_score(X_scaled_full, clusters_4)
fig.add_vline(x=avg_silhouette, line_dash="dash", line_color="red",
              annotation_text=f"Avg: {avg_silhouette:.3f}", row=2, col=2)

# Update layout
fig.update_layout(
    title='Optimal Number of Clusters Analysis',
    width=1400,
    height=900,
    showlegend=True
)

fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=1)
fig.update_yaxes(title_text="Inertia", row=1, col=1)
fig.update_xaxes(title_text="Number of Clusters (k)", row=1, col=2)
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2)
fig.update_xaxes(title_text="Number of Clusters (k)", row=2, col=1)
fig.update_yaxes(title_text="Davies-Bouldin Index", row=2, col=1)
fig.update_xaxes(title_text="Silhouette Coefficient", row=2, col=2)
fig.update_yaxes(title_text="Cluster", row=2, col=2)

fig.show()

print("\n🎯 Optimal k Recommendation:")
print(f"   Elbow Method suggests: k=4 (diminishing returns after this point)")
print(f"   Best Silhouette Score: k={best_k_silhouette} (score={max(silhouette_scores):.3f})")
print(f"   Best Davies-Bouldin: k={k_range[np.argmin(davies_bouldin_scores)]} (score={min(davies_bouldin_scores):.3f})")
print(f"\n💡 Recommendation: Use k=4 for this patient cohort")
print("   This balances cluster quality with interpretability")

🔍 Evaluating different numbers of clusters...
   k=2: Inertia=1133.31, Silhouette=0.469
   k=3: Inertia=625.22, Silhouette=0.515
   k=4: Inertia=454.89, Silhouette=0.475
   k=5: Inertia=396.84, Silhouette=0.455
   k=6: Inertia=354.84, Silhouette=0.393
   k=7: Inertia=326.40, Silhouette=0.355


   k=8: Inertia=309.52, Silhouette=0.340


   k=9: Inertia=291.76, Silhouette=0.337


   k=10: Inertia=275.22, Silhouette=0.338



🎯 Optimal k Recommendation:
   Elbow Method suggests: k=4 (diminishing returns after this point)
   Best Silhouette Score: k=3 (score=0.515)
   Best Davies-Bouldin: k=3 (score=0.747)

💡 Recommendation: Use k=4 for this patient cohort
   This balances cluster quality with interpretability


## 7. Care Pathway Application

Discovered phenotypes are only useful if they mean something over time. A patient is not
permanently a member of one cohort - they move between them as they age and accumulate
conditions.

**Scenario**: we follow a patient who starts out young and healthy. Left alone, the usual
path runs **young and healthy → metabolic → complex chronic**. The question a care team
actually has to answer is *when* to intervene, and the phenotype plane makes that visible:
the transition out of the metabolic cohort is where a program still changes the
trajectory, and where it stops being cheap to wait.


In [8]:
# Use our trained K-Means model to lay out the phenotype landscape
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_final = kmeans_final.fit_predict(X_scaled_full)
patient_data['cluster'] = clusters_final

# Get centroid characteristics
centroids_final = scaler_full.inverse_transform(kmeans_final.cluster_centers_)

# Create the care pathway map
fig = go.Figure()

# Plot phenotypes with different colours
phenotype_interpretations = []

for i in range(4):
    cohort_data = patient_data[patient_data['cluster'] == i]

    # Interpret the cohort from its centroid
    avg_age = centroids_final[i, 0]
    avg_burden = centroids_final[i, 1]
    avg_hba1c = centroids_final[i, 2]
    avg_egfr = centroids_final[i, 3]

    if avg_burden > 5.5:
        cohort_type = "\U0001f534 Complex chronic (case management)"
        rec = "Intensive, multidisciplinary"
        color_override = '#e74c3c'
    elif avg_age > 75:
        cohort_type = "\U0001f7e3 Frail elderly (comprehensive assessment)"
        rec = "Frailty review and falls prevention"
        color_override = '#9b59b6'
    elif avg_burden > 1.4:
        cohort_type = "\U0001f7e0 Metabolic (modifiable risk)"
        rec = "Intervene here - the trajectory is still reversible"
        color_override = '#e67e22'
    else:
        cohort_type = "\U0001f535 Young and healthy (low contact)"
        rec = "Routine primary care only"
        color_override = '#3498db'

    phenotype_interpretations.append({
        'phenotype': i + 1,
        'type': cohort_type,
        'recommendation': rec,
        'avg_age': avg_age,
        'avg_burden': avg_burden,
        'avg_hba1c': avg_hba1c,
        'avg_egfr': avg_egfr
    })

    fig.add_trace(
        go.Scatter(
            x=cohort_data['comorbidity_burden'],
            y=cohort_data['age'],
            mode='markers',
            name=f'Phenotype {i+1}: {cohort_type}',
            marker=dict(size=8, color=color_override, opacity=0.45),
            hovertemplate=
                '<b>Phenotype %{text}</b><br>' +
                'Comorbidity burden: %{x:.1f}<br>' +
                'Age: %{y:.0f} years<br>' +
                f'Cohort mean HbA1c: {avg_hba1c:.2f} %<br>' +
                f'Cohort mean eGFR: {avg_egfr:.0f} mL/min/1.73m2<br>' +
                f'<b>{rec}</b>' +
                '<extra></extra>',
            text=[f'{i+1}' for _ in range(len(cohort_data))]
        )
    )

# ---- The care pathway: how one patient moves through phenotype space over time ----
# Each waypoint is (comorbidity burden, age) - the plane clinicians reason on.
pathway_points = [
    (0.4, 28, 'Age 28<br>young and healthy'),
    (1.3, 42, 'Age 42<br>first condition'),
    (2.4, 52, 'Age 52<br>metabolic'),
    (4.3, 60, 'Age 60<br>multimorbidity'),
    (6.8, 68, 'Age 68<br>complex chronic'),
]

fig.add_trace(
    go.Scatter(
        x=[p[0] for p in pathway_points],
        y=[p[1] for p in pathway_points],
        mode='lines+markers+text',
        name='\u274c Usual trajectory (no intervention)',
        line=dict(color='black', width=4),
        marker=dict(size=14, symbol='circle', color='black'),
        text=[p[2] for p in pathway_points],
        textposition='middle right',
        textfont=dict(size=10, color='black')
    )
)

# The alternative trajectory: the same patient, intervened on in the metabolic cohort
intervened_points = [
    (2.4, 52),
    (3.0, 60),
    (3.4, 68),
    (3.8, 76),
]

fig.add_trace(
    go.Scatter(
        x=[p[0] for p in intervened_points],
        y=[p[1] for p in intervened_points],
        mode='lines+markers',
        name='\u2705 With intervention at the metabolic stage',
        line=dict(color='#16a085', width=4, dash='dash'),
        marker=dict(size=12, symbol='diamond', color='#16a085')
    )
)

# Mark the point where intervening still changes the trajectory
fig.add_trace(
    go.Scatter(
        x=[2.4],
        y=[52],
        mode='markers+text',
        name='Intervention window',
        marker=dict(size=26, color='#f1c40f', symbol='star',
                    line=dict(color='black', width=2)),
        text=['INTERVENTION WINDOW'],
        textposition='bottom center',
        textfont=dict(size=12, color='#b7950b')
    )
)

# Update layout
fig.update_layout(
    title='Care Pathway Through Discovered Phenotype Space',
    xaxis_title='Comorbidity burden (number of active conditions)',
    yaxis_title='Age (years)',
    width=1400,
    height=700,
    showlegend=True,
    hovermode='closest'
)

fig.show()

# Print care pathway recommendations
print("\n\U0001fa7a Care Recommendations Based on Discovered Phenotypes:")
print("=" * 80)
for info in phenotype_interpretations:
    print(f"\nPhenotype {info['phenotype']}: {info['type']}")
    print(f"  Average age: {info['avg_age']:.1f} years")
    print(f"  Average comorbidity burden: {info['avg_burden']:.2f} conditions")
    print(f"  Average HbA1c: {info['avg_hba1c']:.2f} %")
    print(f"  Average eGFR: {info['avg_egfr']:.0f} mL/min/1.73m2")
    print(f"  Recommendation: {info['recommendation']}")

print("\n\U0001f4a1 Care Pathway Insights:")
print("  • Patients do not sit in one cohort - they migrate up and to the right over time")
print("  • The metabolic cohort is the last point where the trajectory is cheaply reversible")
print("  • Intervening there keeps a patient out of the complex chronic cohort for years")
print("  • Waiting until the complex chronic cohort means managing conditions, not preventing them")
print("\n  This is the practical value of unsupervised learning: the cohorts were never")
print("  labelled, yet they tell us where in a patient's life a service should act.")



🩺 Care Recommendations Based on Discovered Phenotypes:

Phenotype 1: 🟣 Frail elderly (comprehensive assessment)
  Average age: 81.0 years
  Average comorbidity burden: 4.58 conditions
  Average HbA1c: 6.16 %
  Average eGFR: 45 mL/min/1.73m2
  Recommendation: Frailty review and falls prevention

Phenotype 2: 🔵 Young and healthy (low contact)
  Average age: 28.1 years
  Average comorbidity burden: 0.49 conditions
  Average HbA1c: 5.23 %
  Average eGFR: 105 mL/min/1.73m2
  Recommendation: Routine primary care only

Phenotype 3: 🟠 Metabolic (modifiable risk)
  Average age: 52.9 years
  Average comorbidity burden: 2.23 conditions
  Average HbA1c: 7.86 %
  Average eGFR: 84 mL/min/1.73m2
  Recommendation: Intervene here - the trajectory is still reversible

Phenotype 4: 🔴 Complex chronic (case management)
  Average age: 66.9 years
  Average comorbidity burden: 6.77 conditions
  Average HbA1c: 7.45 %
  Average eGFR: 39 mL/min/1.73m2
  Recommendation: Intensive, multidisciplinary

💡 Care Pathw

## 8. Summary and Best Practices

### 🎯 What We've Learned

**Unsupervised Learning Fundamentals:**
- **Pattern discovery**: algorithms find structure in unlabelled data
- **No ground truth needed**: discovers natural cohorts automatically
- **K-Means algorithm**: iteratively optimizes cluster assignments
- **Visual learning process**: watched centroids migrate to optimal positions

**Key Differences from Supervised Learning:**

| Aspect | Supervised Learning | Unsupervised Learning |
|--------|---------------------|----------------------|
| Labels | Required | Not required |
| Goal | Predict known outcomes | Discover hidden patterns |
| Evaluation | Accuracy, precision, recall | Inertia, silhouette score |
| Example | Readmission prediction | Patient phenotype discovery |

### 🩺 K-Means Clustering Strengths & Limitations

**Strengths:**
- ✅ **Fast and scalable**: works across a whole registered population
- ✅ **Easy to interpret**: a centroid is a description of the typical patient in a cohort
- ✅ **Versatile**: applicable to many domains (population health, customer segmentation, image compression)
- ✅ **Simple concept**: easy to explain to a clinical audience

**Limitations:**
- ❌ **Requires k specification**: you must choose the number of cohorts (use the elbow method!)
- ❌ **Assumes spherical clusters**: struggles with irregular shapes
- ❌ **Sensitive to initialization**: can converge to local optima (use multiple runs)
- ❌ **Sensitive to outliers**: a handful of extreme patients can drag a centroid

### 🔧 Best Practices for Population Health Applications

**1. Feature Selection**
- Include clinically meaningful variables (age, comorbidity burden, key labs, utilization)
- Normalize/standardize features (they live on wildly different scales)
- Consider adding temporal features (rate of change, not just a single snapshot)
- Check that a feature is not simply a proxy for access to care

**2. Choosing k (Number of Phenotypes)**
- Use the elbow method as a starting point
- Validate with silhouette analysis
- Consider clinical knowledge (how many distinct cohorts can a service realistically run?)
- Balance granularity against interpretability

**3. Model Validation**
- Compare discovered cohorts with recognized clinical phenotypes
- Validate with clinicians before the cohorts drive any care decision
- Test on new patients to make sure the structure generalizes
- Check cluster stability over time, and across sites

**4. Practical Implementation**
- Refresh the clustering as new data arrives; cohorts drift as a population ages
- Provide a confidence score alongside each cohort assignment
- Audit for disparities: a cohort that maps onto deprivation or ethnicity needs explaining, not deploying
- Never let a cohort label override an individual clinical judgement

### 🚀 Next Steps and Extensions

**Immediate Applications:**
1. **Service planning**: size programs to the cohorts that actually exist
2. **Care pathway design**: target interventions at the transition points
3. **Multi-year trends**: track how cohorts shift as a population ages
4. **Risk stratification**: identify who is about to move into a higher-cost cohort

**Advanced Techniques to Explore:**
1. **DBSCAN**: for irregularly shaped cohorts and rare-disease outliers
2. **Hierarchical Clustering**: build a taxonomy of phenotypes
3. **GMM (Gaussian Mixture Models)**: soft assignments, since patients sit between cohorts
4. **Time-series clustering**: discover trajectory shapes, not just snapshots
5. **Ensemble clustering**: combine several algorithms for robustness

### 📊 Real-World Impact

**Population Health:**
- Programs aimed at the cohort that will actually benefit
- Earlier intervention at the point the trajectory is still reversible
- Fewer unplanned admissions from the highest-utilization cohort
- Capacity planned against the population you have, not the one you assumed

**Clinical Research:**
- Automated discovery of phenotypes within a heterogeneous diagnosis
- Identification of subgroups that respond differently to treatment
- Trial recruitment targeted at a defined phenotype
- Hypothesis generation from routinely collected data

**Other Applications:**
- **Customer Segmentation**: group customers by behaviour
- **Image Compression**: reduce colours while preserving quality
- **Anomaly Detection**: identify unusual patterns in data
- **Document Clustering**: organize large text collections

### 💡 Key Takeaways

1. **Unsupervised learning discovers patterns without labels** - perfect when ground truth is unknown or expensive
2. **K-Means is intuitive and visual** - you can watch the algorithm learn
3. **Choosing k is both art and science** - elbow method + silhouette + clinical knowledge
4. **Cohorts mean most longitudinally** - the value is in the transitions between them
5. **Interpretability matters** - a centroid gives a cohort a description a clinician can recognize

---

**You now understand how unsupervised learning automatically discovers hidden patterns in data!**
This foundation prepares you for more advanced clustering techniques and real-world applications.


In [9]:
print("\U0001f389 Patient Phenotype Discovery Tutorial Complete! \U0001f389")
print("\n\U0001f4ca Key Achievements:")
print("  \u2705 Visualized how K-Means discovers patterns through centroid migration")
print("  \u2705 Understood unsupervised learning vs supervised learning")
print("  \u2705 Learned to choose optimal k using elbow method and silhouette analysis")
print("  \u2705 Applied clustering to care pathway design")
print("  \u2705 Interpreted discovered phenotypes with clinical knowledge")

print("\n\U0001fa7a Population Health Applications Covered:")
print("  \U0001f9ec Automatic phenotype discovery from routine measurements")
print("  \U0001f5fa\ufe0f Care pathway design using discovered cohorts")
print("  \u23f1\ufe0f Intervention timing at the point a trajectory is still reversible")
print("  \U0001f4c8 Cluster quality evaluation and validation")

print("\n\U0001f680 You're now ready to:")
print("  \u2022 Apply unsupervised learning to your own datasets")
print("  \u2022 Discover hidden patterns in unlabeled data")
print("  \u2022 Choose and tune clustering algorithms")
print("  \u2022 Evaluate cluster quality objectively")
print("  \u2022 Communicate insights to stakeholders")

print("\n\U0001f393 Next topics to explore:")
print("  \u2022 DBSCAN for irregular cluster shapes")
print("  \u2022 Hierarchical clustering for taxonomies")
print("  \u2022 Dimensionality reduction (PCA, t-SNE)")
print("  \u2022 Gaussian Mixture Models for soft clustering")
print("  \u2022 Real-world applications in your domain")


🎉 Patient Phenotype Discovery Tutorial Complete! 🎉

📊 Key Achievements:
  ✅ Visualized how K-Means discovers patterns through centroid migration
  ✅ Understood unsupervised learning vs supervised learning
  ✅ Learned to choose optimal k using elbow method and silhouette analysis
  ✅ Applied clustering to care pathway design
  ✅ Interpreted discovered phenotypes with clinical knowledge

🩺 Population Health Applications Covered:
  🧬 Automatic phenotype discovery from routine measurements
  🗺️ Care pathway design using discovered cohorts
  ⏱️ Intervention timing at the point a trajectory is still reversible
  📈 Cluster quality evaluation and validation

🚀 You're now ready to:
  • Apply unsupervised learning to your own datasets
  • Discover hidden patterns in unlabeled data
  • Choose and tune clustering algorithms
  • Evaluate cluster quality objectively
  • Communicate insights to stakeholders

🎓 Next topics to explore:
  • DBSCAN for irregular cluster shapes
  • Hierarchical clustering